# KNN and K-means — pure NumPy

**Dataset:** `iris.csv` (150 rows, 4 measurements, 3 species).

Both algorithms depend on distance, so standardisation is part of the algorithm, not decoration. Fit the scaler on training rows only.

## 1. Load the data

In [ ]:
import numpy as np

raw = np.genfromtxt('iris.csv', delimiter=',', names=True)
feature_names = list(raw.dtype.names[:-1])
X = np.column_stack([raw[name] for name in feature_names]).astype(float)
y = np.asarray(raw['species'], dtype=int)

print('rows:', X.shape[0], '| features:', X.shape[1])
print('class counts:', np.bincount(y).tolist())

## 2. Stratified split and train-only scaling

In [ ]:
rng = np.random.default_rng(3)
train_idx, test_idx = [], []
for label in range(3):
    idx = np.where(y == label)[0]
    rng.shuffle(idx)
    cut = int(0.75 * len(idx))
    train_idx.extend(idx[:cut])
    test_idx.extend(idx[cut:])
train_idx = np.array(train_idx)
test_idx = np.array(test_idx)

mean = X[train_idx].mean(axis=0)
scale = X[train_idx].std(axis=0)
scale[scale == 0] = 1.0
Xtr = (X[train_idx] - mean) / scale
Xte = (X[test_idx] - mean) / scale
ytr, yte = y[train_idx], y[test_idx]
print('train:', len(ytr), '| test:', len(yte))

## 3. KNN — compute every test/train distance, then vote

In [ ]:
def knn_predict(X_train, y_train, X_test, k):
    distances = ((X_test[:, None, :] - X_train[None, :, :]) ** 2).sum(axis=2)
    nearest = np.argsort(distances, axis=1)[:, :k]
    votes = y_train[nearest]
    return np.array([np.bincount(row, minlength=3).argmax() for row in votes])

for k in (1, 3, 5, 7, 9):
    pred = knn_predict(Xtr, ytr, Xte, k)
    print(f'k={k}: accuracy={np.mean(pred == yte):.3f}')

## 4. K-means — Lloyd's algorithm with a fixed seed

In [ ]:
def kmeans(X_data, k, seed=5, iterations=100):
    rng = np.random.default_rng(seed)
    centers = X_data[rng.choice(len(X_data), size=k, replace=False)].copy()
    labels = np.zeros(len(X_data), dtype=int)
    for step in range(iterations):
        distances = ((X_data[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
        new_labels = distances.argmin(axis=1)
        new_centers = centers.copy()
        for cluster in range(k):
            members = X_data[new_labels == cluster]
            if len(members):
                new_centers[cluster] = members.mean(axis=0)
        if np.allclose(new_centers, centers):
            labels = new_labels
            break
        centers = new_centers
        labels = new_labels
    inertia = float(((X_data - centers[labels]) ** 2).sum())
    return labels, centers, inertia, step + 1

all_scaled = (X - X[train_idx].mean(axis=0)) / np.where(X[train_idx].std(axis=0) == 0, 1, X[train_idx].std(axis=0))
labels, centers, inertia, steps = kmeans(all_scaled, 3)
print('iterations:', steps, '| inertia:', round(inertia, 2))
print('cluster sizes:', np.bincount(labels, minlength=3).tolist())

## 5. Cluster purity — clusters are not labels

In [ ]:
purity = 0
for cluster in range(3):
    members = y[labels == cluster]
    if len(members):
        counts = np.bincount(members, minlength=3)
        purity += int(counts.max())
        print(f'cluster {cluster}: size={len(members)} | class counts={counts.tolist()}')
print('purity:', round(purity / len(y), 3))
print('K-means found groups; the species names are what your labels say, not what the algorithm knows.')

## 6. Limitations

- KNN stores the training set; prediction cost grows with `n`, and irrelevant features distort distance.
- K-means assumes roughly spherical clusters, needs `k` in advance, and can settle in a local optimum. The seed and initial centres matter.
- Purity looks good when clusters are balanced; it can hide a cluster that mixes two classes or a class split across clusters.
- Both experiments use scaled features, but scaling does not make a feature meaningful — domain choices still decide.

**Exercise:** run K-means with k=2 and k=4. Compare inertia and purity, then explain why a lower inertia is not automatically a better answer.